In [ ]:
# CELL 1: Setup and Imports
import os
import sys
import json
import matplotlib.pyplot as plt
 
sys.path.append('..')
 
from configs.config import Config
from data.splits import get_loaders
from models.bu_net import BUNet
from training.trainer import Trainer
from configs.metrics import evaluate
 
Config.create_dirs()
print(f"✓ Using device: {Config.DEVICE}")
print(f"✓ Data path: {Config.TRAIN_DATASET_PATH}")

✓ Using device: mps
✓ Data path: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData


In [ ]:
# CELL 2: Data Loading
train_loader, val_loader, test_loader = get_loaders(Config)
 
# Sanity check
sample_batch = next(iter(train_loader))
print(f"✓ Image batch shape: {sample_batch['image'].shape}")
print(f"✓ Mask batch shape:  {sample_batch['mask'].shape}")

Splits -> Train: 258, Val: 74, Test: 37
✓ Valid patients: 258/258
✓ Valid patients: 74/74
✓ Valid patients: 37/37
✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463
✓ Image batch shape: torch.Size([8, 2, 128, 128])
✓ Mask batch shape: torch.Size([8, 128, 128])


In [ ]:
# CELL 3: Model + Training
model = BUNet(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)
 
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")
 
trainer = Trainer(model, Config, train_loader, val_loader)
history = trainer.train()

Using device: mps
Epoch [1/35] Batch [0/3225] Loss: 0.8948
Epoch [1/35] Batch [50/3225] Loss: 0.6172
Epoch [1/35] Batch [100/3225] Loss: 0.6230
Epoch [1/35] Batch [150/3225] Loss: 0.6291
Epoch [1/35] Batch [200/3225] Loss: 0.3476
Epoch [1/35] Batch [250/3225] Loss: 0.3788
Epoch [1/35] Batch [300/3225] Loss: 0.3859
Epoch [1/35] Batch [350/3225] Loss: 0.3347
Epoch [1/35] Batch [400/3225] Loss: 0.3073
Epoch [1/35] Batch [450/3225] Loss: 0.6291
Epoch [1/35] Batch [500/3225] Loss: 0.3857
Epoch [1/35] Batch [550/3225] Loss: 0.3169
Epoch [1/35] Batch [600/3225] Loss: 0.5295
Epoch [1/35] Batch [650/3225] Loss: 0.3442
Epoch [1/35] Batch [700/3225] Loss: 0.1979
Epoch [1/35] Batch [750/3225] Loss: 0.1759
Epoch [1/35] Batch [800/3225] Loss: 0.2785
Epoch [1/35] Batch [850/3225] Loss: 0.4502
Epoch [1/35] Batch [900/3225] Loss: 0.3590
Epoch [1/35] Batch [950/3225] Loss: 0.2823
Epoch [1/35] Batch [1000/3225] Loss: 0.4328
Epoch [1/35] Batch [1050/3225] Loss: 0.2399
Epoch [1/35] Batch [1100/3225] Loss: 

KeyboardInterrupt: 

In [ ]:
# CELL 4: Evaluate on Test Set
results = evaluate(model, test_loader, Config.DEVICE)

print(f"\n{'='*50}")
print("TEST SET RESULTS")
print(f"{'='*50}")
print(f"  Dice  Background : {results['dice_class_0']:.4f}")
print(f"  Dice  NCR (1)    : {results['dice_class_1']:.4f}")
print(f"  Dice  Edema (2)  : {results['dice_class_2']:.4f}")
print(f"  Dice  ET  (3)    : {results['dice_class_3']:.4f}")
print(f"  Mean Tumor Dice  : {results['mean_tumor_dice']:.4f}  ← key metric")
print(f"  HD95  NCR (1)    : {results['hd95_class_1']:.2f} px")
print(f"  HD95  Edema (2)  : {results['hd95_class_2']:.2f} px")
print(f"  HD95  ET  (3)    : {results['hd95_class_3']:.2f} px")
print(f"  Mean Tumor HD95  : {results['mean_tumor_hd95']:.2f} px")
print(f"{'='*50}")

results_path = os.path.join(Config.RESULTS_DIR, 'baseline_test_metrics.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"✓ Saved to: {results_path}")

Evaluating on test set...

TEST SET RESULTS
  Dice  Background : 0.9952
  Dice  NCR (1)    : 0.6968
  Dice  Edema (2)  : 0.6669
  Dice  ET  (3)    : 0.7369
  Mean Tumor Dice  : 0.7002  ← key metric
  HD95  NCR (1)    : 1.97 px
  HD95  Edema (2)  : 6.00 px
  HD95  ET  (3)    : 5.23 px
  Mean Tumor HD95  : 4.18 px


In [ ]:
# ── Cell 5: Training Curves ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Dice Loss')
ax.set_title('Baseline U-Net Training')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'baseline_training_curve.png'), dpi=150)
plt.show()